In [ ]:
! pip install shap


! pip install ipywidgets
! jupyter nbextension enable --py widgetsnbextension


In [ ]:
import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import os
import random

from PIL import Image
import numpy as np

import shap

from ipywidgets import FloatProgress

## Load the data

In [ ]:
data_dir = 'F:/Liver_Fibrosis_US_processed'

In [ ]:
# Print 10 sample images from each class
classes = os.listdir(data_dir)


num_classes = len(classes)
fig, axes = plt.subplots(num_classes, 10, figsize=(20, 2 * num_classes))


for i, cls in enumerate(classes):
    cls_path = os.path.join(data_dir, cls)
    images = os.listdir(cls_path)[:10]
    for j, img_name in enumerate(images):
        img_path = os.path.join(cls_path, img_name)
        img = mpimg.imread(img_path)
        axes[i, j].imshow(img, cmap='gray')
        axes[i, j].axis('off')
    axes[i, 0].set_ylabel(cls, size='large')

plt.show()

## Load the model

In [ ]:
print("\n" + "="*80)
print("LOADING TRAINED MODELS")
print("="*80)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load DenseNet201 model for US images
densenet_model_path = '../Models/Ultrasound/Densenet201/densenet201_full_model.pt'
print(f"\nLoading DenseNet201 from: {densenet_model_path}")
densenet_model = torch.load(densenet_model_path, map_location=device, weights_only=False)
densenet_model.eval()
print("✅ DenseNet201 loaded successfully")

In [ ]:
print(densenet_model)

## Get prediction from the model

In [ ]:
image = os.path.join(data_dir, 'F1', 'a13.jpg')


img = mpimg.imread(image)
plt.imshow(img, cmap='gray')

In [ ]:
# Convert to numpy array
arr = np.array(img)

# If grayscale, duplicate channels to make 3 channels
if arr.ndim == 2:
    arr = np.stack([arr, arr, arr], axis=-1)

# Discard the alpha channel if present
elif arr.shape[2] == 4:
    arr = arr[:, :, :3]

# Convert to float32 and scale to [0,1]
arr = arr.astype(np.float32)
if arr.max() > 1.0:
    arr /= 255.0


input_tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(device)
test_img_arr = arr

# Inference
with torch.no_grad():
    output = densenet_model(input_tensor)
    pred = int(output.argmax(dim=1).item())

target = 0
target_name = classes[target]
pred_name = classes[pred]
print(f"Target: {target_name} ({target}) | Predicted: {pred_name} ({pred})")

### Explain predictions with SHAP

Build background dataset

In [ ]:
classes = os.listdir(data_dir)
print(f"Classes found: {classes}")

In [ ]:
background = []
num_background = 52

per_class_background = num_background // len(classes)

for cls in classes:
    cls_path = os.path.join(data_dir, cls)
    images = os.listdir(cls_path)
    selected_images = random.sample(images, per_class_background)
    print(f"Selected images for class '{cls}': {selected_images}")
    for img_name in selected_images:
        img_path = os.path.join(cls_path, img_name)
        img = mpimg.imread(img_path)
        
        # Convert to numpy array
        arr = np.array(img)

        # If grayscale, duplicate channels to make 3 channels
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        # Discard the alpha channel if present
        elif arr.shape[2] == 4:
            arr = arr[:, :, :3]

        # Convert to float32 and scale to [0,1]
        arr = arr.astype(np.float32)
        if arr.max() > 1.0:
            arr /= 255.0

        background.append(arr)

print(f"Background dataset size: {len(background)} images")

# Convert background list to a stacked tensor
background = np.array(background)  # Shape: (num_background, H, W, 3)
background_tensor = torch.from_numpy(background).permute(0, 3, 1, 2).to(device)

print(f"Background tensor shape: {background_tensor.shape}")

Fix in place ReLU operations
- DenseNet201 uses in-place ReLU by default
- So the original values get overwritten
- For SHAP, during backpropagation, we need the original values

In [20]:
def disable_inplace_activations(model):
    """Recursively disable inplace operations in ReLU layers"""
    for module in model.modules():
        if isinstance(module, torch.nn.ReLU):
            module.inplace = False
    return model

densenet_model = disable_inplace_activations(densenet_model)
densenet_model.eval()

print("✅ In-place operations disabled for SHAP compatibility")


✅ In-place operations disabled for SHAP compatibility


In [ ]:
e = shap.DeepExplainer(densenet_model, background_tensor)

# Prepare test image
test_img_tensor = torch.from_numpy(test_img_arr[np.newaxis, ...]).permute(0, 3, 1, 2).to(device)
print(f"Test image tensor shape: {test_img_tensor.shape}")

# Get SHAP values
shap_values = e.shap_values(test_img_tensor)

print(f"SHAP values shape: {len(shap_values)} outputs")
for i, sv in enumerate(shap_values):
    print(f"  Class {i} ({classes[i]}): {sv.shape}")